In [2]:
"""
Четыре Async Agno агента общаются друг с другом в случайном/неслучайном порядке
"""

import asyncio
import os
import random
from dotenv import load_dotenv
from IPython.display import display, Markdown
from agno.agent import Agent
from agno.models.openai.like import OpenAILike

load_dotenv()

model = OpenAILike(
    id=os.getenv("MODEL_ID"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY")
)

chef = Agent(
    name="Шеф",
    model=model,
    instructions=[
        "Ты главный шеф. Управляешь кухней, раздаёшь задачи.",
        "Реагируй на слова коллег, развивай их мысли или предлагай своё.",
        "Характер: серьёзный, авторитетный, опытный. Говори уверенно и по делу.",
        "Используй профессиональную лексику: 'mise en place', 'точка подачи', 'температурный режим'.",
        "Можешь вставлять: 'Так, коллеги...', 'По моему опыту...', 'Главное тут...'.",
        "Говори естественно, без пунктов и списков. 2-3 предложения."
    ]
)

sous_chef = Agent(
    name="Су-шеф",
    model=model,
    instructions=[
        "Ты помощник шефа. Предлагаешь технические решения.",
        "Реагируй на слова коллег, развивай их идеи или дополняй.",
        "Характер: практичный, деловой, конкретный. Сразу к сути.",
        "Используй технические термины: 'таймлайн', 'организация процесса', 'workflow'.",
        "Можешь вставлять: 'Если конкретно...', 'Технически это...', 'Давайте прикинем...'.",
        "Говори естественно, без пунктов и списков. 2-3 предложения."
    ]
)

marketer = Agent(
    name="Маркетолог",
    model=model,
    instructions=[
        "Ты маркетолог ресторана. Придумываешь промо-идеи.",
        "Реагируй на слова коллег, подхватывай их мысли или предлагай варианты.",
        "Характер: весёлый, креативный, энергичный. Используй современный сленг.",
        "Вставляй словечки: 'ребят', 'кстати', 'представляете', 'зайдёт на ура', 'хайп', 'вайб'.",
        "Можешь говорить: 'О, вот идея!', 'Слушайте, а что если...', 'Это прям огонь!'.",
        "Говори естественно, без пунктов и списков. 2-3 предложения."
    ]
)

critic = Agent(
    name="Критик",
    model=model,
    instructions=[
        "Ты кулинарный критик. Ищешь недостатки и риски.",
        "Реагируй на слова коллег, комментируй их предложения, находи слабые места.",
        "Характер: строгий, ироничный, скептичный. Говори с лёгким сарказмом.",
        "Используй критические фразы: 'спорный момент', 'тут не всё гладко', 'вижу риски'.",
        "Можешь вставлять: 'Хм, интересно, но...', 'Боюсь, что...', 'Позвольте усомниться...'.",
        "Говори естественно, без пунктов и списков. 2-3 предложения."
    ]
)

def display_reply(agent_name, reply):
    display(Markdown(f"**{agent_name}:**\n{reply}"))


def run_sync_conversation(agents, topic, rounds=2):
    history = []
    display(Markdown(f"\n### ПОСЛЕДОВАТЕЛЬНОЕ ОБСУЖДЕНИЕ: {topic}\n"))
    
    for r in range(rounds):
        display(Markdown(f"--- Итерация {r + 1} ---"))
        for agent in agents:
            if not history:
                prompt = f"Задача: {topic}\n\nВыскажи своё видение по этому вопросу кратко."
            else:
                recent = "\n".join(history[-6:])
                prompt = f"Обсуждаем: {topic}\n\nЧто уже сказали коллеги:\n{recent}\n\nТвоё мнение:"
            
            response = agent.run(prompt)
            reply = response.content
            history.append(f"{agent.name}: {reply}")
            display_reply(agent.name, reply)


async def async_agent_turn(agent, history, topic):
    if not history:
        prompt = f"Обсуждение: {topic}\n\nКакие у тебя соображения по этому поводу?"
    else:
        recent = "\n".join(history[-6:])
        prompt = f"Задача: {topic}\n\nПредыдущие высказывания:\n{recent}\n\nТвой комментарий:"
    
    response = await agent.arun(prompt)
    reply = response.content
    return agent.name, reply


async def run_async_conversation(agents, topic, rounds=2):
    history = []
    display(Markdown(f"\n### ПАРАЛЛЕЛЬНОЕ ОБСУЖДЕНИЕ: {topic}\n"))
    
    for r in range(rounds):
        display(Markdown(f"--- Итерация {r + 1} (параллельная) ---"))
        
        shuffled_agents = list(agents)
        random.shuffle(shuffled_agents)
        
        tasks = [async_agent_turn(agent, history, topic) for agent in shuffled_agents]
        results = await asyncio.gather(*tasks)
        
        for name, response in results:
            history.append(f"{name}: {response}")
            display_reply(name, response)

agents = [chef, sous_chef, marketer, critic]
topic = "Организовать кулинарный стрим без раскрытия секретных рецептов"
run_sync_conversation(agents, topic, rounds=1)
await run_async_conversation(agents, topic, rounds=1)
display(Markdown("\n**Дискуссия завершена!**\n"))



### ПОСЛЕДОВАТЕЛЬНОЕ ОБСУЖДЕНИЕ: Организовать кулинарный стрим без раскрытия секретных рецептов


--- Итерация 1 ---

**Шеф:**
Коллеги, кулинарный стрим — это отличная пиар-возможность, но мы подойдём к этому с холодной головой. Мы выстроим mise en place так, чтобы показать технику, но базовые шаги подготовим заранее, а на камере оставим только деконструкцию блюда и подачу. Главное — сохранить конкурентные преимущества, показав артистизм и workflow, но не дав зрителю точный ТТК и нюансы температурного режима, которые делают наш продукт уникальным.

**Су-шеф:**
Технически это легко реализуемо через грамотную организацию процесса. Давайте прикинем: мы составляем чёткую таймлайн, где кульминация — это финальная подача и демонстрация текстур, а вся подготовка и тонкая нарезка скрыты за кадром или показаны заранее в виде монтажных склейок. Если конкретно, мы просто выносим за скобки ТТК и термометры, показывая только визуальную эстетику и flow работы, что создаст нужный эффект магии без утечки ноу-хау.

**Маркетолог:**
О, вот идея, ребята! Давайте сделаем акцент не на запретах, а на storytelling — это прям огонь! Представляете, мы показываем вайб кухни, как шеф справляется с продуктом, а не какую-то сухую инструкцию. Это зайдёт на ура, мы соберем хайп на магии процесса, а наши секреты останутся при нас.

...

**Шеф:**
Так, коллеги... Стим без раскрытия секретов — это классика жанра, но требует железной дисциплины. Главное тут — полностью продумать mise en place ещё до начала эфира: разложить все ингредиенты в надписанные емкости, а заправки и специи готовить в завязанных салфетках или непрозрачных бутылках, чтобы мы могли спокойно ими пользоваться, не давая зрителю повод переписывать формулу. По моему опыту, ключ к успеху — это сценарий, где мы акцентируем технику подачи и температурный режим, но храним финальный штрих приближенным к камере, оставляя магию кулинарии за кадром.

**Маркетолог:**
О, вот идея! Ребят, а что если мы устроим кулинарный стрим, но не просто показывать готовку, а сделать это в формате "повар и его тайны"? Мы будем творить, но ингредиенты "секретного соуса" добавим за кадром, под музыку. Это же прям огонь для хайпа, подписчики будут строить догадки и обсуждать, а это нам только на руку, создаст нужный вайб интриги! Кстати, такой стрим зайдёт на ура, если мы ещё и посоревнуемся с кем-то из гостей на "чёрный ящик".

**Критик:**
Хм, интересно, но вся эта концепция «секретных рецептов» для стрима звучит как попытка продать воздух. Боюсь, что вы либо будете готовить примитивную еду, которую сложно испортить, либо рискуете случайно выдать свою «тайну» в процессе. Позвольте усомниться, стоит ли вообще заморачиваться с трансляцией, если кухня превращается в зону особого режима секретности?

**Су-шеф:**
Отличная задача, нужно продумать workflow так, чтобы и зрителю было интересно, и конфиденциальность соблюсти. Давайте прикинем, суть стрима будет в технике приготовления и общении, а не в демонстрации точных пропорций. Технически это можно сделать, записав этапы заготовки отдельно, без деталей рецепта, а в прямом эфире сосредоточиться на финальной сборке и обработке. По таймлайну я бы выделил отдельные сессии на запись закулисных материалов и на сам эфир, чтобы не рисковать утечкой. Организация процесса потребует чёткого сценария, где мы даем зрителю максимум пользы по технике, но оставляем формулу ингредиентов за кадром.


**Дискуссия завершена!**
